In [4]:
import nltk
import pandas as pd
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import brown, words, stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.metrics.distance import edit_distance
from nltk.util import ngrams
from nltk.probability import FreqDist

# Download all required NLTK resources
nltk.download('brown', quiet=True)
nltk.download('words', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # This fixes your LookupError
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [5]:
# (a) Load the Brown Corpus and select 20 sentences
# Extracting a block of raw text to demonstrate sentence tokenization properly
raw_text = " ".join(brown.words(categories='news')[:1000])
sentences = sent_tokenize(raw_text)[:20]

stop_words_set = set(stopwords.words('english'))
punctuation_set = set(string.punctuation)

print("--- Q1: Tokenization and Stop Word Removal ---\n")

# Process and display the first 5 sentences as requested
for i in range(5):
    # (b) Sentence and word-level tokenization
    sentence = sentences[i]
    words_tokenized = word_tokenize(sentence)

    # (c) Convert to lowercase & (d) Remove punctuation and stop words
    cleaned_tokens = [
        word.lower() for word in words_tokenized
        if word.lower() not in stop_words_set and word not in punctuation_set and word.isalpha()
    ]

    # (e) Display original and cleaned tokens
    print(f"Sentence {i+1}: {sentence}")
    print(f"Original Word Tokens: {words_tokenized}")
    print(f"Cleaned Tokens: {cleaned_tokens}\n")

--- Q1: Tokenization and Stop Word Removal ---

Sentence 1: The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
Original Word Tokens: ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', 'Atlanta', "'s", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', '``', 'that', 'any', 'irregularities', 'took', 'place', '.']
Cleaned Tokens: ['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlanta', 'recent', 'primary', 'election', 'produced', 'evidence', 'irregularities', 'took', 'place']

Sentence 2: The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .
Original Word Tokens: ['The', 'jury', 'further', 'said', 'in', 'term-end', 

In [6]:
# (a) Extract at least 30 alphabetic words
# Getting 25 diverse words from the corpus
corpus_words = [w.lower() for w in brown.words(categories='news') if w.isalpha()][:25]

# (d) Include the mandatory words requested in the assignment
mandatory_words = ['playing', 'studies', 'running', 'better', 'cars']
target_words = mandatory_words + corpus_words

# (b) Apply Porter Stemmer & (c) Apply WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

comparison_data = []
for word in target_words:
    comparison_data.append({
        'Original Word': word,
        'Stemmed Word (Porter)': stemmer.stem(word),
        'Lemmatized Word (WordNet)': lemmatizer.lemmatize(word, pos='v') # 'v' handles running/playing better
    })

# (e) Create a DataFrame containing original, stemmed, and lemmatized words
df_morphology = pd.DataFrame(comparison_data)

print("--- Q2: Stemming vs Lemmatization Comparison ---\n")
print(df_morphology.to_string(index=False))

--- Q2: Stemming vs Lemmatization Comparison ---

 Original Word Stemmed Word (Porter) Lemmatized Word (WordNet)
       playing                  play                      play
       studies                 studi                     study
       running                   run                       run
        better                better                    better
          cars                   car                      cars
           the                   the                       the
        fulton                fulton                    fulton
        county                counti                    county
         grand                 grand                     grand
          jury                  juri                      jury
          said                  said                       say
        friday                friday                    friday
            an                    an                        an
 investigation              investig             investigation
     

In [7]:
# (a) Load the NLTK Words Corpus (converted to a set for speed)
valid_english_words = set(w.lower() for w in words.words())

def correct_spelling(word):
    word = word.lower()
    if word in valid_english_words:
        return word

    # (c) Use approximate string matching to find corrections
    # Optimize: Only check words starting with the same letter and similar length
    candidates = [w for w in valid_english_words if w.startswith(word[0]) and abs(len(w) - len(word)) <= 2]

    if not candidates:
        return word

    # Find the candidate with the lowest Levenshtein edit distance
    best_match = min(candidates, key=lambda x: edit_distance(word, x))
    return best_match

print("--- Q3: Spelling Correction ---\n")

# (b) Create at least 10 intentionally misspelled words
misspelled_list = ["teh", "computr", "langauge", "processng", "inteligence",
                   "artifical", "natuaral", "universiti", "studentt", "progamming"]

print("10 Misspelled Words & Suggestions:")
for typo in misspelled_list:
    print(f"Misspelled: {typo.ljust(15)} -> Suggestion: {correct_spelling(typo)}")

# (e) Correct at least five example sentences
example_sentences = [
    "i love natuarl langauge processng.",
    "artifical inteligence is the future.",
    "the computr progamming class is hard.",
    "he is a good studentt at the universiti.",
    "teh weather is very nice today."
]

print("\nCorrected Example Sentences:")
for sent in example_sentences:
    tokens = word_tokenize(sent)
    corrected_tokens = [correct_spelling(t) if t.isalpha() else t for t in tokens]
    corrected_sent = " ".join(corrected_tokens)
    print(f"Original:  {sent}")
    print(f"Corrected: {corrected_sent}\n")

--- Q3: Spelling Correction ---

10 Misspelled Words & Suggestions:
Misspelled: teh             -> Suggestion: tew
Misspelled: computr         -> Suggestion: computer
Misspelled: langauge        -> Suggestion: langrage
Misspelled: processng       -> Suggestion: processal
Misspelled: inteligence     -> Suggestion: intelligence
Misspelled: artifical       -> Suggestion: artificial
Misspelled: natuaral        -> Suggestion: natural
Misspelled: universiti      -> Suggestion: university
Misspelled: studentt        -> Suggestion: student
Misspelled: progamming      -> Suggestion: progambling

Corrected Example Sentences:
Original:  i love natuarl langauge processng.
Corrected: i love natuary langrage processal .

Original:  artifical inteligence is the future.
Corrected: artificial intelligence is the future .

Original:  the computr progamming class is hard.
Corrected: the computer progambling class is hard .

Original:  he is a good studentt at the universiti.
Corrected: he is a good stude

In [8]:
# (a) Load a suitable subset of the Brown Corpus & (b) Retain lowercase alphabetic tokens
subset_words = brown.words(categories=['news', 'reviews'])
clean_tokens = [word.lower() for word in subset_words if word.isalpha()]

# (c) Generate unigrams, bigrams, and trigrams
unigrams = clean_tokens
bigrams = list(ngrams(clean_tokens, 2))
trigrams = list(ngrams(clean_tokens, 3))

# (d) Calculate N-gram frequencies
unigram_freq = FreqDist(unigrams)
bigram_freq = FreqDist(bigrams)
trigram_freq = FreqDist(trigrams)

print("--- Q4: N-gram Frequencies ---\n")

# (e) Display the ten most frequent unigrams, bigrams, and trigrams
print("Top 10 Unigrams:")
for item, count in unigram_freq.most_common(10):
    print(f"{item}: {count}")

print("\nTop 10 Bigrams:")
for item, count in bigram_freq.most_common(10):
    print(f"{item}: {count}")

print("\nTop 10 Trigrams:")
for item, count in trigram_freq.most_common(10):
    print(f"{item}: {count}")

--- Q4: N-gram Frequencies ---

Top 10 Unigrams:
the: 8756
of: 4201
and: 3347
a: 3060
to: 2870
in: 2748
for: 1269
is: 1246
that: 1177
was: 944

Top 10 Bigrams:
('of', 'the'): 1197
('in', 'the'): 815
('to', 'the'): 377
('on', 'the'): 326
('for', 'the'): 290
('at', 'the'): 274
('and', 'the'): 242
('with', 'the'): 202
('of', 'a'): 190
('that', 'the'): 186

Top 10 Trigrams:
('one', 'of', 'the'): 72
('the', 'united', 'states'): 39
('members', 'of', 'the'): 31
('some', 'of', 'the'): 29
('as', 'well', 'as'): 24
('president', 'of', 'the'): 23
('a', 'number', 'of'): 22
('of', 'the', 'new'): 21
('the', 'new', 'york'): 21
('as', 'a', 'result'): 20
